In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

FUNCTION-FUNCTION PENTING

In [9]:
# 1. Fungsi Pembersihan Genre
def clean_genres(genre_string):
    if pd.isna(genre_string) or not isinstance(genre_string, str):
        return []
    # Ganti '&' dan ',' dengan spasi, ubah ke lowercase
    cleaned = genre_string.lower().replace('&', ' ').replace(',', ' ')
    words = cleaned.split()
    # Buang kata-kata sampah yang berulang
    stop_words = {'tv', 'shows', 'movies', 'show', 'movie'}
    filtered_words = [w for w in words if w not in stop_words]
    return filtered_words

In [10]:

# 2. Fungsi Cek Semua Genre Unik di Dataset
def get_all_unique_genres(dataframe):
    unique_genres = set()
    for listed_in in dataframe['listed_in'].dropna():
        unique_genres.update(clean_genres(listed_in))
    return sorted(list(unique_genres))

In [11]:

# 3. Fungsi Batasan Rating Umur (Sama seperti bawaanmu)
def get_max_allowed_rating(age):
    if age < 7: return ['G', 'TV-Y', 'TV-G']
    elif age < 13: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG']
    elif age < 14: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13']
    elif age < 17: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13', 'TV-14']
    else: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13', 'TV-14', 'R', 'NC-17', 'TV-MA', 'NR', 'UR']

In [ ]:
# 4. Fungsi Utama Rekomendasi (Mengatasi Cold-Start & Fallback)
def get_recommendations(user_data, df_catalog, embeddings_matrix, top_n=5):
    age = user_data.get('age', 18)
    watch_history = user_data.get('watch_history', [])
    preferred_genres = user_data.get('preferred_genres', [])
    
    # Filter rating umur di awal agar aman
    allowed_ratings = get_max_allowed_rating(age)
    df_scores = df_catalog.copy()
    df_scores['rating'] = df_scores['rating'].fillna('NR')
    df_scores = df_scores[df_scores['rating'].isin(allowed_ratings)].reset_index(drop=True)
    
    # Ambil ulang matriks embedding yang sudah difilter berdasarkan indeks rating umur
    # (indeks catalog yang valid sesuai rating umur)
    valid_indices = df_catalog[df_catalog['rating'].fillna('NR').isin(allowed_ratings)].index
    filtered_embeddings = embeddings_matrix[valid_indices]
    
    # Bersihkan preferensi genre user (abaikan kata yang tidak ada di list token unik)
    catalog_genres = get_all_unique_genres(df_catalog)
    clean_user_pref = [w for genre in preferred_genres for w in clean_genres(genre) if w in catalog_genres]
    
    # Cari indeks film history di dataframe yang sudah difilter umur
    history_titles = [m['title'].lower() for m in watch_history]
    history_indices_in_filtered = df_scores[df_scores['title'].str.lower().isin(history_titles)].index.tolist()
    
    # --- LOGIKA REKOMENDASI & HANDLING COLD-START ---
    if not watch_history or not history_indices_in_filtered:
        # TANTANGAN: COLD START (History Kosong / Film Tidak Ditemukan)
        # Fallback: Buat profile vector berdasarkan 'preferred_genres' murni menggunakan model text
        if clean_user_pref:
            pref_text = " ".join(clean_user_pref)
            # Encode teks preferensi menggunakan model global yang sedang aktif
            user_profile_vector = model.encode([pref_text]).reshape(1, -1)
            scores = cosine_similarity(user_profile_vector, filtered_embeddings)[0]
        else:
            # Jika preferred_genres juga kosong/tidak valid, fallback ke item terbaru di katalog yang aman
            return df_scores.sort_values('release_year', ascending=False).head(top_n)
    else:
        # KONDISI NORMAL: Ambil rata-rata embedding dari history nonton
        user_vecs = filtered_embeddings[history_indices_in_filtered]
        user_profile_vector = np.mean(user_vecs, axis=0).reshape(1, -1)
        scores = cosine_similarity(user_profile_vector, filtered_embeddings)[0]
    
    df_scores['similarity_score'] = scores
    
    # Hapus film yang sudah ditonton agar tidak direkomendasikan lagi
    df_final = df_scores[~df_scores['title'].str.lower().isin(history_titles)]
    return df_final.sort_values('similarity_score', ascending=False).head(top_n)

In [13]:
def evaluate_recommendations(recommendations, user_data, df_catalog):
    if recommendations.empty:
        return { "hit_rate": 0, "f1_score": 0, "precision_at_k": 0, "dcg": 0.0, "ndcg": 0.0, "relevance_scores": [], "ideal_relevance_scores": [] }

    # ── Bangun kumpulan kata target dari preferred genres + watch history ──
    user_target_words = set()
    for g in user_data.get('preferred_genres', []):
        user_target_words.update(clean_genres(g))

    for item in user_data.get('watch_history', []):
        match = df_catalog[df_catalog['title'].str.lower() == item['title'].lower()]
        if not match.empty:
            row = match.iloc[0]
            user_target_words.update(clean_genres(row['listed_in']))
            if row['director']:
                user_target_words.update(row['director'].lower().replace(',', ' ').split())
            if row['cast']:
                user_target_words.update(row['cast'].lower().replace(',', ' ').split())

    hits = 0
    f1_scores = []
    relevance_scores = []   # Graded relevance per posisi (0-3 scale)

    for _, row in recommendations.iterrows():
        rec_words = set(clean_genres(row['listed_in']))
        if row['director']:
            rec_words.update(row['director'].lower().replace(',', ' ').split())
        if row['cast']:
            rec_words.update(row['cast'].lower().replace(',', ' ').split())

        intersection = user_target_words.intersection(rec_words)

        # ── Hit ──
        if len(intersection) > 0:
            hits += 1

        # ── F1 ──
        if len(rec_words) == 0 or len(user_target_words) == 0:
            f1 = 0.0
        else:
            precision = len(intersection) / len(rec_words)
            recall    = len(intersection) / len(user_target_words)
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)

        # ── Graded Relevance (0-3) berdasarkan overlap ratio ──
        if len(user_target_words) == 0:
            rel = 0
        else:
            overlap_ratio = len(intersection) / len(user_target_words)
            if overlap_ratio >= 0.15:
                rel = 3
            elif overlap_ratio >= 0.08:
                rel = 2
            elif overlap_ratio > 0:
                rel = 1
            else:
                rel = 0
        relevance_scores.append(rel)

    k = len(relevance_scores)

    # ── Precision@K ──
    precision_at_k = hits / k if k > 0 else 0.0

    # ── DCG  (formula 9 dari slide: rel_i / log2(i+1)) ──
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevance_scores))

    # ── IDCG – ideal ordering (sort descending) ──
    ideal_rels = sorted(relevance_scores, reverse=True)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))

    # ── NDCG ──
    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    hit_rate = 1 if hits > 0 else 0

    return {
        "hit_rate":              hit_rate,
        "f1_score":              round(float(np.mean(f1_scores)), 4),
        "precision_at_k":        round(float(precision_at_k), 4),
        "dcg":                   round(float(dcg), 4),
        "ndcg":                  round(float(ndcg), 4),
        "relevance_scores":      relevance_scores,
        "ideal_relevance_scores": ideal_rels,
    }

LOAD DATA

In [14]:
df = pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


PRE-PROCESSING DATA

In [16]:
# Ambil kolom yang dibutuhkan dan tangani NaN
required_features = ['title', 'rating', 'listed_in', 'description', 'director', 'cast']
for feature in required_features:
    df[feature] = df[feature].fillna('')

# Bikin kolom teks gabungan dengan menerapkan pembersihan listed_in langsung di stringnya
def build_combined_text(row):
    # Ganti nama variabel lokal menjadi 'genres_text' atau 'genres_cleaned'
    genres_text = " ".join(clean_genres(row['listed_in']))
    
    # Gunakan variabel baru tersebut di f-string
    return f"{row['title']}{row['director']}{row['cast']}{genres_text} {row['description']}".lower()

df['combined_features'] = df.apply(build_combined_text, axis=1)
print("Data Preprocessing Selesai. Kolom 'combined_features' siap di-embed.")

Data Preprocessing Selesai. Kolom 'combined_features' siap di-embed.


In [17]:
# Ambil seluruh genre pada 'listed_in' untuk referensi (bisa digunakan untuk validasi preferensi genre user)
all_genres = get_all_unique_genres(df)
print(f"Total genre unik yang ditemukan: {len(all_genres)}")
print("Daftar genre unik:")
for genre in all_genres:
    print("-", genre, "\b")

Total genre unik yang ditemukan: 39
Daftar genre unik:
- action
- adventure
- anime
- british
- children
- classic
- comedies
- comedy
- crime
- cult
- documentaries
- docuseries
- dramas
- faith
- family
- fantasy
- features
- horror
- independent
- international
- kids'
- korean
- lgbtq
- music
- musicals
- mysteries
- nature
- reality
- romantic
- sci-fi
- science
- series
- spanish-language
- spirituality
- sports
- stand-up
- talk
- teen
- thrillers


INISIALISASI METODE EMBEDDING (SBERT)

In [18]:
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

print("Generating embeddings... Mohon tunggu.")
embeddings = model.encode(df['combined_features'].tolist(), show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

Generating embeddings... Mohon tunggu.


Batches:   0%|          | 0/276 [00:00<?, ?it/s]

Embeddings shape: (8807, 384)


Pengujian dengan mock metadata

In [20]:
# Simulasi Data User dari app.py milikmu
users_metadata = {
    "U001": {
        "name": "Arya Kusuma",
        "age": 24,
        "preferred_genres": ["Thrillers", "Sci-Fi", "Fantasy"],
        "watch_history": [
            {"title": "Accident"},
            {"title": "Cam"},
            {"title": "Edge of Fear"},
            {"title": "Gaddar: the Traitor"},
            {"title": "Grandmaster"},
            {"title": "Apostle"},
            {"title": "My Friend Pinto"},
            {"title": "Animal World"},
            {"title": "How It Ends"},
            {"title": "Anon"},
            {"title": "Orbiter 9"},
            {"title": "Beyond Skyline"},
            {"title": "Star Trek: The Next Generation"},
            {"title": "The Titan"},
        ]
    },
    # COLD START TEST CASE (History kosong, preferred_genre aman)
    "U002": {
        "name": "User Baru Cold Start",
        "age": 20,
        "preferred_genres": ["Comedies", "Action"],
        "watch_history": [] 
    }
}

# JALANKAN PENGUJIAN UNTUK USER ARYA (U001)
target_user = users_metadata["U001"]

print(f"=== HASIL REKOMENDASI UNTUK: {target_user['name']} ===")
hasil_recs = get_recommendations(target_user, df, embeddings, top_n=5)
display(hasil_recs[['title', 'listed_in', 'similarity_score']])

print(f"\n=== HASIL METRIK EVALUASI: {target_user['name']} ===")
metrik = evaluate_recommendations(hasil_recs, target_user, df)
df_metrik = pd.Series(metrik, name="Nilai")
display(df_metrik)

print("\n")

# JALANKAN PENGUJIAN UNTUK USER COLD START (U002)
target_user = users_metadata["U002"]

print(f"=== HASIL REKOMENDASI UNTUK: {target_user['name']} ===")
hasil_recs = get_recommendations(target_user, df, embeddings, top_n=5)
display(hasil_recs[['title', 'listed_in', 'similarity_score']])

print(f"\n=== HASIL METRIK EVALUASI: {target_user['name']} ===")
metrik = evaluate_recommendations(hasil_recs, target_user, df)
df_metrik = pd.Series(metrik, name="Nilai")
display(df_metrik)

=== HASIL REKOMENDASI UNTUK: Arya Kusuma ===


,title,listed_in,similarity_score
5113,Bright,"Action & Adventure, Sci-Fi & Fantasy",0.721860
3072,Riot,Action & Adventure,0.697993
1584,Ava,"Action & Adventure, Dramas",0.695497
8671,Vincent N Roxxy,"Dramas, Thrillers",0.694902
4028,Into the Badlands,TV Action & Adventure,0.693001



=== HASIL METRIK EVALUASI: Arya Kusuma ===


hit_rate                                1
f1_score                           0.0301
precision_at_k                        1.0
dcg                                2.9485
ndcg                                  1.0
relevance_scores          [1, 1, 1, 1, 1]
ideal_relevance_scores    [1, 1, 1, 1, 1]
Name: Nilai, dtype: object



=== HASIL REKOMENDASI UNTUK: User Baru Cold Start ===


,title,listed_in,similarity_score
2144,GAME ON: A Comedy Crossover Event,"Kids' TV, TV Comedies",0.557825
618,America: The Motion Picture,"Action & Adventure, Comedies",0.554756
3898,Lunatics,"International TV Shows, TV Comedies",0.551750
4570,Hot Date,"Romantic TV Shows, TV Comedies",0.546514
1038,Dancing Angels,"International TV Shows, Romantic TV Shows, TV ...",0.527347



=== HASIL METRIK EVALUASI: User Baru Cold Start ===


hit_rate                                1
f1_score                           0.2445
precision_at_k                        1.0
dcg                                8.8454
ndcg                                  1.0
relevance_scores          [3, 3, 3, 3, 3]
ideal_relevance_scores    [3, 3, 3, 3, 3]
Name: Nilai, dtype: object